In [1]:
import numpy as np
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
def load_data(id: int):
    data = fetch_ucirepo(id = id)

    name = data.metadata.name
    X = data.data.features
    y = data.data.targets

    return name, X, y

In [6]:
def eda(X, y):
    print(f"Feature set shape : {X.shape}")
    print(f"Target set shape : {y.shape}")
    print("=" * 50)

    print("\nNull Values")
    print(X.isna().sum())
    print(y.isna().sum())

    print("=" * 50)

    print("\nColumns")
    print(X.columns.tolist())

    print("=" * 50)
    print("Class Distribution")
    print(y.value_counts())

In [20]:
class LogisticRegression:
    def __init__(self, learning_rate = 0.01, epochs = 1000):
        self.W = None
        self.b = 0.0
        self.lr = learning_rate
        self.epochs = epochs
        self.cost_history = None

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def predict_proba(self, X):
        z = self.linear_model(X)
        return self.sigmoid(z)

    def predict(self, X, threshold = 0.5):
        probability = self.predict_proba(X)
        return (probability >= threshold).astype(int)

    def linear_model(self, X):
        return X @ self.W + self.b

    def calculate_loss(self, y_pred, y):
        e = 1e-10
        y_pred = np.clip(y_pred, e, 1-e)

        loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
        return loss

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1) 
        
        n_samples, n_features = X.shape

        self.W = np.zeros((n_features, 1))
        self.b = 0.0
        self.cost_history = []
        
        for epoch in range(self.epochs):
            z = self.linear_model(X)
            prediction = self.sigmoid(z)
            error = prediction - y

            dw = (1 / n_samples) * (X.T @ error)
            db = (1 / n_samples) * np.sum(error)

            self.W -= self.lr * dw
            self.b -= self.lr * db

            loss = self.calculate_loss(prediction, y)
            if epoch % 100 == 0:
                print(f"Epoch : {epoch}, Loss : {loss:.4f}")
                self.cost_history.append(loss)

        
        return self


In [11]:
def evaluate(y_act, y_pred):
    y_act = np.asarray(y_act)
    y_pred = np.asarray(y_pred)

    TP = np.sum((y_act == 1) & (y_pred == 1))
    TN = np.sum((y_act == 0) & (y_pred == 0))
    FP = np.sum((y_act == 0) & (y_pred == 1))
    FN = np.sum((y_act == 1) & (y_pred == 0))

    accuracy = (TP + TN) / len(y_act)
    precision = TP / (TP + FP) if (TP + FP) != 0 else 1
    recall = TP / (TP + FN) if (TP + FN) != 0 else 1
    f1 = (2 * precision * recall) / (precision + recall)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1-score': f1
    }

In [21]:
name, X, y = load_data(id=267)
print(f"Exploratory data analysis for {name} dataset")
eda(X, y)

print("=" * 50)
print("Logistic Regression from scratch")
print("=" * 50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(learning_rate=0.01, epochs=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(f"Weights: {model.W}")
print(f"Bias: {model.b}")

print("=" * 50)
print("Evaluation Metrics")
for key, value in evaluate(y_test, y_pred).items():
    print(f"{key}: {value:.4f}")



Exploratory data analysis for Banknote Authentication dataset
Feature set shape : (1372, 4)
Target set shape : (1372, 1)

Null Values
variance    0
skewness    0
curtosis    0
entropy     0
dtype: int64
class    0
dtype: int64

Columns
['variance', 'skewness', 'curtosis', 'entropy']
Class Distribution
class
0        762
1        610
Name: count, dtype: int64
Logistic Regression from scratch
Epoch : 0, Loss : 0.6931
Epoch : 100, Loss : 0.5541
Epoch : 200, Loss : 0.4770
Epoch : 300, Loss : 0.4268
Epoch : 400, Loss : 0.3904
Epoch : 500, Loss : 0.3620
Epoch : 600, Loss : 0.3390
Epoch : 700, Loss : 0.3196
Epoch : 800, Loss : 0.3030
Epoch : 900, Loss : 0.2884
Weights: [[-1.53184138]
 [-0.8880082 ]
 [-0.39261103]
 [-0.0034243 ]]
Bias: -0.2692648315350613
Evaluation Metrics
accuracy: 0.9018
precision: 0.9386
recall: 0.8425
f1-score: 0.8880
